In [ ]:
# GradCAM Saliency Export & Quantification
#
# Computes GradCAM saliency maps for retinal fundus images using a ViT-based eGFR
# prediction model, and quantifies attention with respect to retinal structures.
#
# Pipeline:
#   1. Load model checkpoint (ViT backbone + eGFR regression head)
#   2. Run GradCAM (target: vit.blocks[-1].norm1) over the full test set
#   3. Save origin images, saliency tensors, and metadata per run
#   4. Generate representative averaged heatmaps (Left / Right eye separately)
#      with FOV masking and in-FOV 0-1 renormalization
#   5. Quantify saliency within retinal structures (artery, vein, optic disc/cup)
#      and compute attention ratios with bootstrap CIs
#
# Outputs (per architecture × fine-tuning config):
#   TENSOR_ROOT/<desc>/  — origin_images_gradcam.pth, saliency_map_gradcam.pth, meta_gradcam.json
#   REP_ROOT/<desc>/     — Left/Right heatmap and overlay PNGs
#   TENSOR_ROOT/<desc>/  — RESULTS_test_all_gradcam.csv

import os, sys, math, json, random, warnings, glob
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as tfms
import torchvision.transforms.functional as TF
from tqdm import tqdm

import matplotlib.pyplot as plt


# -------------------------------------------------------------------------
# Paths — update these to match your environment
# -------------------------------------------------------------------------
# WORK_DIR   : project root; must contain models/encoder/__init__.py with build_model
# OUT_DIR    : directory where inference output CSVs (infout_test_*.csv) are stored
# ROOT_CKPT  : directory containing per-run checkpoint folders (ckpt.pth.tar)
# MASTER_CSV : master CSV with columns including 'pngfilename' and 'eye_orientation'
# IMG_DIR    : base directory for fundus PNG images
# TENSOR_ROOT: output directory for saved GradCAM tensors (.pth) and metadata
# REP_ROOT   : output directory for representative heatmap images
# ARTERY_DIR : binary artery segmentation masks (AutoMorph M2 output)
# VEIN_DIR   : binary vein segmentation masks (AutoMorph M2 output)
# ODC_RAW_DIR: raw optic disc/cup segmentation images (AutoMorph M2 output, R/B channels)
# -------------------------------------------------------------------------
WORK_DIR    = "/path/to/project/root"
OUT_DIR     = "/path/to/inference/outputs"
ROOT_CKPT   = "/path/to/checkpoints"
MASTER_CSV  = "/path/to/master.csv"
IMG_DIR     = "/path/to/fundus/images/"
TENSOR_ROOT = Path("/path/to/gradcam/tensors")
REP_ROOT    = Path("/path/to/representative/heatmaps")
ARTERY_DIR  = Path("/path/to/artery_binary_masks")
VEIN_DIR    = Path("/path/to/vein_binary_masks")
ODC_RAW_DIR = Path("/path/to/optic_disc_cup_raw")

if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
os.chdir(WORK_DIR)

TENSOR_ROOT.mkdir(exist_ok=True, parents=True)
REP_ROOT.mkdir(exist_ok=True, parents=True)

IMG_SIZE = 448
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Fine-tuning configurations to iterate over
FT_SPECS = [
    dict(ft="lora",    ft_blks="full", lora_rank=4),
    dict(ft="partial", ft_blks=1,      lora_rank=None),
    dict(ft="linear",  ft_blks=None,   lora_rank=None),
]

DEBUG_SHOW_RANDOM_HEATMAPS = True
DEBUG_N_SHOW = 2
DEBUG_SEED   = 42


# -------------------------------------------------------------------------
# Model definition
# -------------------------------------------------------------------------
from models.encoder import build_model


class EgfrOnlyModel(nn.Module):
    """ViT backbone + single linear head for eGFR regression."""

    def __init__(self, base_model, feat_dim=None, init_mu=None):
        super().__init__()
        self.base = base_model

        D = feat_dim
        if D is None:
            D = (getattr(self.base, "num_features", None)
                 or getattr(self.base, "embed_dim", None)
                 or getattr(getattr(self.base, "head", nn.Identity()), "in_features", None)
                 or 768)

        self.head_egfr = nn.Linear(D, 1)

        if init_mu is not None:
            with torch.no_grad():
                self.head_egfr.bias.data.fill_(init_mu)
                self.head_egfr.weight.data.zero_()

    def forward_features(self, x):
        feats = None
        if hasattr(self.base, "forward_features"):
            feats = self.base.forward_features(x)
        else:
            try:
                feats = self.base.forward(x, return_features=True)
            except TypeError:
                feats = self.base(x)

        if isinstance(feats, dict):
            preferred = (
                "x_norm_clstoken", "cls_token", "pooled", "last_hidden_state",
                "features", "x_norm_patchtokens", "patch_tokens", "tokens",
            )
            chosen = None
            for k in preferred:
                if k in feats and feats[k] is not None:
                    chosen = feats[k]
                    break
            if chosen is None:
                for v in feats.values():
                    if torch.is_tensor(v):
                        chosen = v
                        break
            if chosen is None:
                raise RuntimeError("Could not extract tensor features from dict output.")
            feats = chosen

        if torch.is_tensor(feats) and feats.ndim == 3:
            feats = feats[:, 0]
        return feats

    def forward(self, x):
        f = self.forward_features(x)
        return {"egfr": self.head_egfr(f).squeeze(1)}


def build_full_model(arch, ft, ft_blks=None, lora_rank=None, img_size=IMG_SIZE):
    args = type("Args", (), {})()
    args.arch     = arch
    args.ft       = ft
    args.img_size = img_size
    if ft in ["partial", "lora"]:
        args.ft_blks = ft_blks
    if ft == "lora":
        args.lora_rank = int(lora_rank) if lora_rank is not None else 4
    base  = build_model(args).to(device)
    model = EgfrOnlyModel(base_model=base).to(device)
    return model.eval()


def load_ckpt(model, ckpt_path):
    sd = torch.load(ckpt_path, map_location="cpu")
    if isinstance(sd, dict) and any(k.startswith("base.") or k.startswith("head_egfr") for k in sd):
        model.load_state_dict(sd, strict=True)
        return True
    for key in ["state_dict", "model", "net", "module", "ema_state"]:
        if isinstance(sd, dict) and key in sd and isinstance(sd[key], dict):
            try:
                model.load_state_dict(sd[key], strict=True)
                return True
            except Exception:
                pass
    try:
        model.load_state_dict(sd, strict=False)
        return True
    except Exception as e:
        print(f"[FAIL] load_ckpt: {ckpt_path} | {e}")
        return False


# -------------------------------------------------------------------------
# Naming helpers
# Checkpoint folder names and inference-output filenames follow slightly
# different conventions (e.g. "rank_4" vs "rank4"), so these are kept separate.
# -------------------------------------------------------------------------
def ckpt_desc_from_spec(arch, ft, ft_blks=None, lora_rank=None):
    if ft == "lora":    return f"{arch}_lora_rank_{int(lora_rank)}_ft_{ft_blks}"
    if ft == "partial": return f"{arch}_partial_ft_{int(ft_blks)}"
    if ft == "linear":  return f"{arch}_linear"
    return f"{arch}_{ft}"


def infout_suffix_from_spec(arch, ft, ft_blks=None, lora_rank=None):
    if ft == "lora":    return f"{arch}_lora_rank{int(lora_rank)}_ft{ft_blks}"
    if ft == "partial": return f"{arch}_partial_ft{int(ft_blks)}"
    if ft == "linear":  return f"{arch}_linear"
    return f"{arch}_{ft}"


def ckpt_path_from_ckpt_desc(ckpt_desc):
    return str(Path(ROOT_CKPT) / ckpt_desc / "ckpt.pth.tar")


def find_infout_test_csv_by_suffix(suffix):
    cand = Path(OUT_DIR) / f"infout_test_{suffix}_egfr_trainer.csv"
    if cand.exists():
        return str(cand)
    gl = glob.glob(str(Path(OUT_DIR) / f"infout_test_*{suffix}*_egfr_trainer.csv"))
    return gl[0] if gl else None


# -------------------------------------------------------------------------
# Master CSV — used only to attach eye_orientation labels
# -------------------------------------------------------------------------
def sanitize(fn):
    b, e = os.path.splitext(str(fn).strip())
    return b.replace(".", "_") + e


def load_master_for_eye(master_csv=MASTER_CSV):
    if not Path(master_csv).exists():
        print(f"[WARN] MASTER_CSV not found: {master_csv}")
        return pd.DataFrame(columns=["png_base", "eye_orientation"])
    df = pd.read_csv(master_csv, low_memory=False)
    if "pngfilename" not in df.columns:
        return pd.DataFrame(columns=["png_base", "eye_orientation"])
    df = df.copy()
    df["pngfilename_sanitized"] = df["pngfilename"].astype(str).apply(sanitize)
    df["img_path"]  = IMG_DIR + df["pngfilename_sanitized"]
    df["png_base"]  = df["img_path"].apply(lambda p: os.path.basename(str(p)))
    if "eye_orientation" not in df.columns:
        df["eye_orientation"] = np.nan
    return df[["png_base", "eye_orientation"]].drop_duplicates()


master_eye = load_master_for_eye(MASTER_CSV)


# -------------------------------------------------------------------------
# ViT GradCAM
# Target layer: vit.blocks[-1].norm1
# -------------------------------------------------------------------------
def find_vit_backbone(module):
    if hasattr(module, "blocks"):
        return module
    for _, child in module.named_children():
        vb = find_vit_backbone(child)
        if vb is not None:
            return vb
    return None


class ViTGradCAM:
    """Gradient-weighted class activation mapping for ViT-based models."""

    def __init__(self, model, target_module):
        self.model          = model.eval().to(device)
        self.target_module  = target_module
        self.activations    = None
        self._activations_var = None
        self.handle = self.target_module.register_forward_hook(self._fwd_hook)

    def _fwd_hook(self, module, inp, out):
        self.activations = out.detach()
        out.retain_grad()
        self._activations_var = out

    def remove(self):
        if self.handle is not None:
            self.handle.remove()
            self.handle = None

    def _tokens_to_grid(self, tokens):
        """Reshape [B, N, D] token sequence to [B, D, H, W] spatial grid.
        Handles cls token and optional register tokens automatically."""
        B, N, D = tokens.shape
        for extra in [1, 2, 5, 6, 9, 0]:
            n_patch = N - extra
            if n_patch <= 0:
                continue
            side = int(round(math.sqrt(n_patch)))
            if side * side == n_patch:
                break
        else:
            side  = int(math.floor(math.sqrt(N - 1)))
            extra = N - side * side

        patch_tokens = tokens[:, extra:, :]
        return patch_tokens.reshape(B, side, side, D).permute(0, 3, 1, 2).contiguous()

    def compute_gradcam(self, x, out_size=(IMG_SIZE, IMG_SIZE)):
        self.model.zero_grad(set_to_none=True)
        with torch.enable_grad():
            out  = self.model(x)
            loss = out["egfr"].sum()
            loss.backward()

        grads = self._activations_var.grad.detach()
        feat  = self._tokens_to_grid(self.activations)
        grad  = self._tokens_to_grid(grads)

        weights = grad.flatten(2).mean(dim=2).unsqueeze(-1).unsqueeze(-1)
        cam = F.relu((weights * feat).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=out_size, mode="bicubic", align_corners=False).squeeze(1)

        mn  = cam.flatten(1).min(dim=1)[0].view(-1, 1, 1)
        mx  = cam.flatten(1).max(dim=1)[0].view(-1, 1, 1)
        cam = (cam - mn) / (mx - mn + 1e-12)

        return cam.detach().cpu()


# -------------------------------------------------------------------------
# Tensor export — run GradCAM over the full test set and save results
# -------------------------------------------------------------------------
preprocess = tfms.Compose([tfms.Resize((IMG_SIZE, IMG_SIZE)), tfms.ToTensor()])


def load_tensor_01(img_path):
    return preprocess(Image.open(img_path).convert("RGB"))


def compute_cam_tensor(cam_engine, img_path):
    if not isinstance(img_path, str) or not os.path.exists(img_path):
        return None, None
    x   = load_tensor_01(img_path).unsqueeze(0).to(device, non_blocking=True)
    sal = cam_engine.compute_gradcam(x, out_size=(IMG_SIZE, IMG_SIZE))
    return x.squeeze(0).detach().cpu(), sal[0].float().clamp(0, 1).unsqueeze(0)


def export_test_all_pack(test_df, out_dir, cam_engine):
    os.makedirs(out_dir, exist_ok=True)
    ori_list, cam_list, meta_list = [], [], []

    for i, row in tqdm(test_df.iterrows(), total=len(test_df), desc=Path(out_dir).name):
        img_path = str(row["pngfilename"])
        ori, cam = compute_cam_tensor(cam_engine, img_path)
        if ori is None:
            continue
        ori_list.append(ori.unsqueeze(0))
        cam_list.append(cam.unsqueeze(0))
        meta_list.append({
            "row_idx":         int(i),
            "pngfilename":     img_path,
            "png_base":        os.path.basename(img_path),
            "eye_orientation": row.get("eye_orientation", None),
            "pred_egfr":       float(row.get("pred_egfr", np.nan)),
            "true_egfr":       float(row.get("true_egfr", np.nan)),
        })

    if not ori_list:
        print(f"[WARN] No images exported: {out_dir}")
        return False

    torch.save(torch.cat(ori_list).contiguous(), os.path.join(out_dir, "origin_images_gradcam.pth"))
    torch.save(torch.cat(cam_list).contiguous(), os.path.join(out_dir, "saliency_map_gradcam.pth"))
    with open(os.path.join(out_dir, "meta_gradcam.json"), "w") as f:
        json.dump(meta_list, f, indent=2)
    test_df.to_csv(Path(out_dir) / "test_df_gradcam.csv", index=False)
    print(f"[✓] Saved: {out_dir} (N={len(meta_list)})")
    return True


# -------------------------------------------------------------------------
# Debug visualization — show random samples inline (no file save)
# -------------------------------------------------------------------------
def _to_numpy_img(t):
    return t.detach().cpu().clamp(0, 1).permute(1, 2, 0).numpy()


def _overlay_cam(rgb_hw3, cam_hw, alpha=0.45, cmap_name="jet"):
    cam_rgb = plt.get_cmap(cmap_name)(cam_hw)[..., :3]
    return np.clip((1 - alpha) * rgb_hw3 + alpha * cam_rgb, 0, 1)


def show_random_heatmaps_from_saved_pack(out_dir, n_show=2, seed=2, cmap="jet"):
    p   = Path(out_dir)
    ori = torch.load(p / "origin_images_gradcam.pth")
    cam = torch.load(p / "saliency_map_gradcam.pth")
    with open(p / "meta_gradcam.json") as f:
        metas = json.load(f)

    N    = min(len(ori), len(cam), len(metas))
    idxs = random.Random(seed).sample(range(N), k=min(n_show, N))
    print(f"[DEBUG] Showing heatmaps for indices: {idxs}")

    fig, axes = plt.subplots(len(idxs), 3, figsize=(14, 4 * len(idxs)))
    if len(idxs) == 1:
        axes = np.expand_dims(axes, 0)

    for r, idx in enumerate(idxs):
        rgb   = _to_numpy_img(ori[idx])
        hm    = cam[idx, 0].detach().cpu().numpy()
        title = metas[idx].get("png_base", f"idx={idx}")
        for ax, img, ttl in zip(axes[r], [rgb, hm, _overlay_cam(rgb, hm)],
                                ["Origin", "GradCAM", "Overlay"]):
            ax.imshow(img, cmap=(cmap if ttl == "GradCAM" else None))
            ax.set_title(f"{title}\n({ttl})")
            ax.axis("off")
    plt.tight_layout()
    plt.show()


# -------------------------------------------------------------------------
# Representative heatmaps — averaged per eye side (L / R)
# FOV masking + in-FOV 0-1 renormalization applied before averaging
# -------------------------------------------------------------------------
def _norm_eye_label(x):
    if x is None:
        return None
    s = str(x).strip().lower()
    if s in ["r", "right", "rt", "od", "o.d", "1", "true"]:
        return "R"
    if s in ["l", "left", "lt", "os", "o.s", "0", "false"]:
        return "L"
    try:
        return "R" if int(float(s)) == 1 else "L"
    except Exception:
        return None


def _load_pack(src_dir):
    p   = Path(src_dir)
    ori = torch.load(p / "origin_images_gradcam.pth")
    cam = torch.load(p / "saliency_map_gradcam.pth")
    with open(p / "meta_gradcam.json") as f:
        metas = json.load(f)
    return ori, cam, metas


def _indices_by_eye(metas, eye_code):
    return [i for i, m in enumerate(metas) if _norm_eye_label(m.get("eye_orientation")) == eye_code]


def make_fov_mask_np(h=IMG_SIZE, w=IMG_SIZE):
    yy, xx = np.ogrid[:h, :w]
    cy, cx = h // 2, w // 2
    r = min(h, w) // 2
    return ((yy - cy) ** 2 + (xx - cx) ** 2 <= r ** 2).astype(np.float32)


def _apply_fov_and_renorm_01(mean_hw, fov01):
    if mean_hw is None:
        return None
    m      = mean_hw.astype(np.float32) * fov01
    inside = m[fov01 > 0]
    if inside.size == 0:
        return None
    mn, mx = float(np.min(inside)), float(np.max(inside))
    if mx <= mn + 1e-12:
        return m
    return ((m - mn) / (mx - mn + 1e-12)) * fov01


def _avg_heatmap_fov_renorm01(cam, indices, fov01):
    if not indices:
        return None
    mean = cam[indices, 0].detach().cpu().numpy().mean(axis=0)
    return _apply_fov_and_renorm_01(mean, fov01)


def save_rep_lr(src_dir, desc, set_name="TEST", seed=123, cmap="jet"):
    ori, cam, metas = _load_pack(src_dir)
    fov01 = make_fov_mask_np()

    idxs_L = _indices_by_eye(metas, "L")
    idxs_R = _indices_by_eye(metas, "R")
    avg_L  = _avg_heatmap_fov_renorm01(cam, idxs_L, fov01)
    avg_R  = _avg_heatmap_fov_renorm01(cam, idxs_R, fov01)

    rL = random.Random(seed).choice(idxs_L)   if idxs_L else None
    rR = random.Random(seed + 1).choice(idxs_R) if idxs_R else None

    out_dir = REP_ROOT / f"{desc}_gradcam"
    out_dir.mkdir(exist_ok=True, parents=True)

    for avg, idx, side in [(avg_L, rL, "Left"), (avg_R, rR, "Right")]:
        if avg is not None:
            plt.imsave(out_dir / f"{set_name}_{side}_heatmap_gradcam.png", avg, cmap=cmap)
            if idx is not None:
                ov = _overlay_cam(_to_numpy_img(ori[idx]), avg, cmap_name=cmap)
                plt.imsave(out_dir / f"{set_name}_{side}_overlay_gradcam.png", ov)

    print(f"[✓] Representative saved: {out_dir}")


# -------------------------------------------------------------------------
# Segmentation mask utilities
# -------------------------------------------------------------------------
def ensure_bool_mask(mask_img, size=(IMG_SIZE, IMG_SIZE)):
    if not isinstance(mask_img, Image.Image):
        mask_img = Image.fromarray(np.asarray(mask_img))
    mask_np = np.array(mask_img.resize(size, resample=Image.NEAREST))
    return torch.from_numpy((mask_np > 0).astype(np.uint8))


def load_mask_by_filename(root, pngfilename):
    base = os.path.basename(str(pngfilename))
    cand = root / base
    if cand.exists():
        return ensure_bool_mask(Image.open(cand))
    stem = Path(base).stem
    for ext in (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"):
        p = root / f"{stem}{ext}"
        if p.exists():
            return ensure_bool_mask(Image.open(p))
    return None


def load_optic_disc_cup_union(pngfilename):
    """Load ODC mask: union of red (disc) and blue (cup) channels."""
    base = os.path.basename(str(pngfilename))
    p    = ODC_RAW_DIR / base
    if not p.exists():
        stem = Path(base).stem
        p = next((ODC_RAW_DIR / f"{stem}{e}"
                  for e in (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")
                  if (ODC_RAW_DIR / f"{stem}{e}").exists()), None)
        if p is None:
            return None
    try:
        arr = np.array(Image.open(p).convert("RGB"))
        union = ((arr[..., 0] > 0) | (arr[..., 2] > 0)).astype(np.uint8)
        return ensure_bool_mask(Image.fromarray(union * 255))
    except Exception:
        return None


def split_odc_quadrants(odc):
    """Split the ODC region into superior / inferior / nasal / temporal quadrants."""
    if odc is None or odc.ndim != 2:
        return None
    ys, xs = torch.nonzero(odc > 0, as_tuple=True)
    if ys.numel() == 0:
        return None
    cy, cx = ys.float().mean(), xs.float().mean()
    h, w   = odc.shape
    yy = torch.arange(h).view(-1, 1).expand(h, w).float()
    xx = torch.arange(w).view(1, -1).expand(h, w).float()
    s1 = (yy - cy) - (xx - cx)
    s2 = (yy - cy) + (xx - cx)
    return {
        "superior": ((s1 < 0) & (s2 < 0) & (odc > 0)).to(torch.uint8),
        "inferior": ((s1 > 0) & (s2 > 0) & (odc > 0)).to(torch.uint8),
        "right":    ((s1 < 0) & (s2 > 0) & (odc > 0)).to(torch.uint8),
        "left":     ((s1 > 0) & (s2 < 0) & (odc > 0)).to(torch.uint8),
    }


def make_fov_mask(h=IMG_SIZE, w=IMG_SIZE):
    yy, xx = np.ogrid[:h, :w]
    cy, cx = h // 2, w // 2
    r = min(h, w) // 2
    return torch.from_numpy(((yy - cy) ** 2 + (xx - cx) ** 2 <= r ** 2).astype(np.uint8))


# -------------------------------------------------------------------------
# GradCAM quantification against retinal structures
# -------------------------------------------------------------------------
def normalize_gradcam_into_fov(gradcam_t, fov_mask):
    """Resize GradCAM to IMG_SIZE, mask to FOV, normalize so sum = 100."""
    if gradcam_t.dim() == 3:
        gradcam_t = gradcam_t[0]
    elif gradcam_t.dim() == 4:
        gradcam_t = gradcam_t.squeeze()
    gc = TF.pil_to_tensor(
        TF.to_pil_image(gradcam_t.float()).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    ).squeeze().float()
    gc = gc * fov_mask
    s  = gc.sum().item()
    return (100.0 * gc / s) if s > 0 else None


def region_mean(gc_norm100, region_mask01):
    denom = region_mask01.sum().item()
    return float("nan") if denom <= 0 else (gc_norm100 * region_mask01).sum().item() / denom


def compute_metrics_for_dataset(df_csv, saliency_pth, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    df       = pd.read_csv(df_csv).reset_index(drop=True)
    gradcams = torch.load(saliency_pth)

    n = min(len(df), int(gradcams.shape[0]))
    df       = df.iloc[:n].reset_index(drop=True)
    gradcams = gradcams[:n]

    fov     = make_fov_mask()
    records = []

    for idx in tqdm(range(len(df)), desc=df_csv.stem):
        row        = df.iloc[idx]
        pngfilename = str(row["pngfilename"])

        artery = load_mask_by_filename(ARTERY_DIR, pngfilename)
        vein   = load_mask_by_filename(VEIN_DIR,   pngfilename)
        if artery is None or vein is None:
            continue

        odc  = load_optic_disc_cup_union(pngfilename)
        both = (artery | vein).clamp_(0, 1)

        gc = normalize_gradcam_into_fov(gradcams[idx], fov)
        if gc is None:
            continue

        vein_   = (vein   * fov).clamp_(0, 1)
        artery_ = (artery * fov).clamp_(0, 1)
        both_   = (both   * fov).clamp_(0, 1)
        not_both_ = ((1 - both) * fov).clamp_(0, 1)

        if odc is not None:
            odc = (odc * fov).clamp_(0, 1)
            not_odc           = ((1 - odc)               * fov).clamp_(0, 1)
            not_odc_not_both  = ((1 - (both | odc).clamp_(0, 1)) * fov).clamp_(0, 1)
            quads = split_odc_quadrants(odc)
        else:
            not_odc = not_odc_not_both = quads = None

        eye_code    = _norm_eye_label(row.get("eye_orientation"))
        od_temporal = od_nasal = od_sup = od_inf = float("nan")

        if quads is not None:
            od_sup = region_mean(gc, quads["superior"])
            od_inf = region_mean(gc, quads["inferior"])
            if eye_code == "L":
                t_mask, n_mask = quads["right"], quads["left"]
            elif eye_code == "R":
                t_mask, n_mask = quads["left"],  quads["right"]
            else:
                t_mask = n_mask = None
            if t_mask is not None: od_temporal = region_mean(gc, t_mask)
            if n_mask is not None: od_nasal    = region_mean(gc, n_mask)

        records.append({
            "pngfilename":                str(pngfilename),
            "veins_n":                    region_mean(gc, vein_),
            "not_veins_n":                region_mean(gc, ((1 - vein)   * fov).clamp_(0, 1)),
            "arteries_n":                 region_mean(gc, artery_),
            "not_arteries_n":             region_mean(gc, ((1 - artery) * fov).clamp_(0, 1)),
            "both_n":                     region_mean(gc, both_),
            "not_both_n":                 region_mean(gc, not_both_),
            "has_optic_disc":             bool(odc is not None),
            "optic_disc_n":               float("nan") if odc is None else region_mean(gc, odc),
            "not_optic_disc_n":           float("nan") if odc is None else region_mean(gc, not_odc),
            "not_optic_disc_not_both_n":  float("nan") if odc is None else region_mean(gc, not_odc_not_both),
            "optic_disc_temporal_n":      od_temporal,
            "optic_disc_nasal_n":         od_nasal,
            "optic_disc_superior_n":      od_sup,
            "optic_disc_inferior_n":      od_inf,
        })

    res     = pd.DataFrame.from_records(records)
    out_csv = out_dir / "RESULTS_test_all_gradcam.csv"
    res.to_csv(out_csv, index=False)
    print(f"[Saved] {out_csv} (n={len(res)})")
    return res


def _safe_ratio(num, den):
    num, den = pd.to_numeric(num, errors="coerce"), pd.to_numeric(den, errors="coerce")
    ratio = num / den
    ratio[(~np.isfinite(den)) | (den <= 0)] = np.nan
    return ratio


def bootstrap_ci(data, n_boot=2000, ci=95, seed=42):
    data = np.asarray(data)[np.isfinite(np.asarray(data))]
    if data.size == 0:
        return (np.nan, np.nan)
    rng   = np.random.default_rng(seed)
    means = np.array([np.nanmean(rng.choice(data, size=data.size, replace=True))
                      for _ in range(n_boot)])
    alpha = (100 - ci) / 2.0
    return (np.percentile(means, alpha), np.percentile(means, 100 - alpha))


def summarize_attention(res_df, label="TEST", n_boot=2000):
    """Compute attention ratios relative to background (non-vessel, non-ODC) pixels."""
    ref    = res_df["not_optic_disc_not_both_n"]
    ratios = pd.DataFrame({
        "vascular_attention_vein":           _safe_ratio(res_df["veins_n"],                    ref),
        "vascular_attention_artery":         _safe_ratio(res_df["arteries_n"],                 ref),
        "vascular_attention_both":           _safe_ratio(res_df["both_n"],                     ref),
        "optic_disc_attention":              _safe_ratio(res_df["optic_disc_n"],               ref),
        "optic_disc_attention_temporal":     _safe_ratio(res_df["optic_disc_temporal_n"],      ref),
        "optic_disc_attention_nasal":        _safe_ratio(res_df["optic_disc_nasal_n"],         ref),
        "optic_disc_attention_superior":     _safe_ratio(res_df["optic_disc_superior_n"],      ref),
        "optic_disc_attention_inferior":     _safe_ratio(res_df["optic_disc_inferior_n"],      ref),
    })

    mean_vals = ratios.mean(numeric_only=True)
    out_rows  = {}
    for col in ratios.columns:
        lo, hi = bootstrap_ci(ratios[col].values, n_boot=n_boot)
        out_rows[col] = {
            "mean":        float(mean_vals[col]),
            "95%_CI_low":  float(lo),
            "95%_CI_high": float(hi),
            "n_used":      int(np.isfinite(ratios[col].values).sum()),
        }

    out = pd.DataFrame(out_rows).T
    print(f"\n=== [{label}] Attention ratios (ref = not_optic_disc_not_both_n) ===")
    print(mean_vals.to_string())
    return ratios, out

In [ ]:
# Main loop — iterates over all architecture × fine-tuning combinations and runs the full pipeline:
#   load checkpoint → build GradCAM engine → export tensors → save representative heatmaps
#   → quantify saliency against retinal structures → summarize attention ratios
#
# Filter controls:
#   ALLOW_ARCH    : set of architecture names to process
#   ALLOW_FT      : set of fine-tuning strategies to process (e.g. {"lora"})
#   ALLOW_ARCH_FT : per-architecture override (leave empty to use ALLOW_FT for all)
#   SAMPLE_FRAC   : float in (0, 1] to randomly subsample the test set; None = use all


REQUESTED_ARCHES = ["retfound", "retfound_dinov2", "mae", "openclip", "dinov2", "dinov3"]

ALLOW_ARCH = {"retfound", "retfound_dinov2", "mae", "openclip", "dinov2", "dinov3"}
ALLOW_FT   = {"lora"}

ALLOW_ARCH_FT = {
    # "openclip": {"linear"},
}

SAMPLE_FRAC = None
SEED = 42


def prepare_test_df(infout_csv_path, master_eye_df):
    df = pd.read_csv(infout_csv_path)
    if "pngfilename" not in df.columns:
        raise ValueError(f"'pngfilename' not found in {infout_csv_path}")
    df = df.copy()
    df["png_base"] = df["pngfilename"].astype(str).apply(os.path.basename)
    if len(master_eye_df):
        df = df.merge(master_eye_df, on="png_base", how="left")
    else:
        df["eye_orientation"] = np.nan
    return df


done    = []
skipped = []

for arch in REQUESTED_ARCHES:
    if arch not in ALLOW_ARCH:
        skipped.append((arch, "arch_not_allowed"))
        continue

    for ft_spec in FT_SPECS:
        ft        = ft_spec["ft"]
        ft_blks   = ft_spec["ft_blks"]
        lora_rank = ft_spec["lora_rank"]

        if ft not in ALLOW_FT:
            continue
        if arch in ALLOW_ARCH_FT and ft not in ALLOW_ARCH_FT[arch]:
            continue

        ckpt_desc  = ckpt_desc_from_spec(arch, ft, ft_blks=ft_blks, lora_rank=lora_rank)
        suffix     = infout_suffix_from_spec(arch, ft, ft_blks=ft_blks, lora_rank=lora_rank)
        ckpt       = ckpt_path_from_ckpt_desc(ckpt_desc)
        infout_csv = find_infout_test_csv_by_suffix(suffix)

        if not Path(ckpt).exists() or infout_csv is None:
            skipped.append((f"{arch}|{ft}", "missing_ckpt_or_infout"))
            continue

        SAVE_DESC = suffix
        print("\n" + "=" * 100)
        print(f"[RUN] {SAVE_DESC}  |  arch={arch}  ft={ft}  ft_blks={ft_blks}  lora_rank={lora_rank}")
        print(f"  ckpt       : {ckpt}")
        print(f"  infout_test: {infout_csv}")
        print("=" * 100)

        # load inference output CSV and merge eye orientation labels
        test_df = prepare_test_df(infout_csv, master_eye)

        if SAMPLE_FRAC is not None:
            n_keep  = max(1, int(round(len(test_df) * float(SAMPLE_FRAC))))
            test_df = test_df.sample(n=n_keep, random_state=SEED).reset_index(drop=True)
            print(f"[SAMPLE] keeping {len(test_df)} rows (frac={SAMPLE_FRAC}, seed={SEED})")

        # build model and load checkpoint
        model = build_full_model(arch, ft, ft_blks=ft_blks, lora_rank=lora_rank, img_size=IMG_SIZE)
        if not load_ckpt(model, ckpt):
            print(f"[SKIP] checkpoint load failed: {SAVE_DESC}")
            skipped.append((SAVE_DESC, "ckpt_load_fail"))
            continue

        # locate target layer: last ViT block norm1
        vit = find_vit_backbone(model)
        if vit is None or not hasattr(vit, "blocks") or len(vit.blocks) == 0:
            skipped.append((SAVE_DESC, "no_vit_backbone"))
            continue
        try:
            target_layer = vit.blocks[-1].norm1
        except Exception as e:
            print(f"[SKIP] target layer not accessible: {SAVE_DESC} | {e}")
            skipped.append((SAVE_DESC, "no_target_layer"))
            continue

        cam_engine = ViTGradCAM(model=model, target_module=target_layer)

        # export GradCAM tensors for the full test set
        out_test = str(TENSOR_ROOT / f"{SAVE_DESC}_test_all_gradcam")
        if not export_test_all_pack(test_df, out_test, cam_engine):
            skipped.append((SAVE_DESC, "export_empty"))
            cam_engine.remove()
            continue

        if DEBUG_SHOW_RANDOM_HEATMAPS:
            show_random_heatmaps_from_saved_pack(out_test, n_show=DEBUG_N_SHOW, seed=DEBUG_SEED)

        # save averaged representative heatmaps per eye side
        save_rep_lr(out_test, SAVE_DESC, set_name="TEST", seed=123, cmap="jet")

        # quantify saliency against vascular / ODC masks and summarize
        mask_dir = Path(out_test) / "Mask_gradcam"
        res_df   = compute_metrics_for_dataset(
            Path(out_test) / "test_df_gradcam.csv",
            Path(out_test) / "saliency_map_gradcam.pth",
            mask_dir,
        )
        ratios, summary = summarize_attention(res_df, label=f"TEST:{SAVE_DESC}_gradcam", n_boot=2000)
        summary.to_csv(mask_dir / "ATTN_summary_TEST_gradcam.csv")

        done.append(f"{SAVE_DESC}_gradcam")
        cam_engine.remove()

print("\nDONE")
print(f"  completed : {len(done)}")
print(f"  skipped   : {len(skipped)}")
if done:    print("  runs      :", done[:20], ("..." if len(done)    > 20 else ""))
if skipped: print("  skip log  :", skipped[:20], ("..." if len(skipped) > 20 else ""))

In [ ]:
# Inference output summarizer
#
# Reads all infout_test_*_egfr_trainer.csv files from OUT_DIR and computes
# per-model test-set performance metrics (MSE, MAE, Pearson R, AUROC).
# Results are sorted by fine-tuning method and base model, then saved as a CSV.
#
# Expected columns in each infout CSV: true_egfr, pred_egfr
# Output: model_performance_summary_from_infout_test.csv


import os, glob
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score


# -------------------------------------------------------------------------
# Paths — update to match your environment
# OUT_DIR : directory containing infout_test_*_egfr_trainer.csv files
# SAVE_CSV: path for the output summary CSV
# -------------------------------------------------------------------------
OUT_DIR  = "/path/to/inference/outputs"
SAVE_CSV = os.path.join(OUT_DIR, "model_performance_summary_from_infout_test.csv")

PATTERN = "infout_test_*_egfr_trainer.csv"


# Display order for sorting
FT_ORDER        = ["lora", "partial", "linear"]
BASE_MODEL_ORDER = ["retfound", "retfound_dinov2", "openclip", "mae", "dinov2", "dinov3"]


# -------------------------------------------------------------------------
# Metrics
# -------------------------------------------------------------------------
def pearson_r(a, b):
    a, b = np.asarray(a), np.asarray(b)
    if len(a) < 2 or np.std(a) == 0 or np.std(b) == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def compute_metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)

    def safe_auc(thr):
        y_bin = (y_true < thr).astype(int)
        try:
            return float(roc_auc_score(y_bin, thr - y_pred))
        except Exception:
            return np.nan

    return dict(
        mse           = float(np.mean((y_true - y_pred) ** 2)),
        mae           = float(np.mean(np.abs(y_true - y_pred))),
        mean_pred     = float(np.mean(y_pred)),
        mean_true     = float(np.mean(y_true)),
        pearson_r     = pearson_r(y_true, y_pred),
        auroc_under60 = safe_auc(60),
        auroc_under90 = safe_auc(90),
    )


# -------------------------------------------------------------------------
# Filename parsing — extract base model and fine-tuning method from filename
# -------------------------------------------------------------------------
def infer_desc(fp):
    return (os.path.basename(fp)
            .replace("infout_test_", "")
            .replace("_egfr_trainer.csv", ""))


def parse_base_ft(desc):
    toks = desc.split("_")
    for i, t in enumerate(toks):
        if t in FT_ORDER:
            return "_".join(toks[:i]), t
    return desc, "unknown"


# -------------------------------------------------------------------------
# Main
# -------------------------------------------------------------------------
files = sorted(glob.glob(os.path.join(OUT_DIR, PATTERN)))
if not files:
    raise FileNotFoundError(f"No files matching '{PATTERN}' found in {OUT_DIR}")

rows = []
for fp in files:
    df   = pd.read_csv(fp)
    desc = infer_desc(fp)
    base, ft = parse_base_ft(desc)
    m = compute_metrics(df["true_egfr"].values, df["pred_egfr"].values)
    rows.append({
        "Model":                       base,
        "Fine-tuning method":          ft,
        "Test MSE":                    m["mse"],
        "Test MAE":                    m["mae"],
        "Test avg eGFR prediction":    m["mean_pred"],
        "Test avg true eGFR":          m["mean_true"],
        "Test Pearson R":              m["pearson_r"],
        "Test AUROC (eGFR < 60)":      m["auroc_under60"],
        "Test AUROC (eGFR < 90)":      m["auroc_under90"],
        "_desc":                       desc,
    })

summary = pd.DataFrame(rows)

# sort by fine-tuning method then base model
summary["_ft_rank"]   = summary["Fine-tuning method"].apply(
    lambda x: FT_ORDER.index(x) if x in FT_ORDER else 999)
summary["_base_rank"] = summary["Model"].apply(
    lambda x: BASE_MODEL_ORDER.index(x) if x in BASE_MODEL_ORDER else 999)
summary = summary.sort_values(["_ft_rank", "_base_rank"]).reset_index(drop=True)

# format decimals
fmt = summary.copy()
fmt["Test MSE"]                  = fmt["Test MSE"].map(lambda x: f"{x:.1f}")
fmt["Test MAE"]                  = fmt["Test MAE"].map(lambda x: f"{x:.2f}")
fmt["Test avg eGFR prediction"]  = fmt["Test avg eGFR prediction"].map(lambda x: f"{x:.1f}")
fmt["Test avg true eGFR"]        = fmt["Test avg true eGFR"].map(lambda x: f"{x:.1f}")
fmt["Test Pearson R"]            = fmt["Test Pearson R"].map(lambda x: f"{x:.3f}")
fmt["Test AUROC (eGFR < 60)"]    = fmt["Test AUROC (eGFR < 60)"].map(lambda x: f"{x:.3f}")
fmt["Test AUROC (eGFR < 90)"]    = fmt["Test AUROC (eGFR < 90)"].map(lambda x: f"{x:.3f}")
fmt = fmt.drop(columns=["_ft_rank", "_base_rank", "_desc"])

fmt.to_csv(SAVE_CSV, index=False)
print("Saved →", SAVE_CSV)
display(fmt)

In [ ]:
# Saliency attention ratio summarizer
#
# For each model × fine-tuning combination, reads the per-image GradCAM quantification
# results (RESULTS_test_all_gradcam.csv) and computes attention ratios of retinal
# structures relative to background (non-vessel, non-ODC pixels), reported as
# mean with 95% bootstrap CI.
#
# REP_ROOT subfolder names (e.g. dinov3_lora_rank4_ftfull_gradcam) are used to
# locate matching quantification CSVs under TENSOR_ROOT.
#
# Output columns: Model, Finetuning Method, Non-vessel (ref),
#                 Artery, Vein, Vessel, Optic disc (+ quadrants)
# Output file   : saliency_attention_ratio_summary_95ci.csv


import os, glob, re
import numpy as np
import pandas as pd
from pathlib import Path


# -------------------------------------------------------------------------
# Paths — update to match your environment
# TENSOR_ROOT : directory containing per-run subdirs with RESULTS_test_all_gradcam.csv
# REP_ROOT    : directory containing per-run *_gradcam subdirs (representative heatmaps)
# SAVE_CSV    : path for the output summary CSV
# -------------------------------------------------------------------------
TENSOR_ROOT = Path("/path/to/gradcam/tensors")
REP_ROOT    = Path("/path/to/representative/heatmaps")
SAVE_CSV    = TENSOR_ROOT / "saliency_attention_ratio_summary_95ci.csv"


FT_ORDER         = ["lora", "partial", "linear"]
BASE_MODEL_ORDER = ["retfound", "retfound_dinov2", "openclip", "mae", "dinov2", "dinov3"]

REF_COL = "not_optic_disc_not_both_n"

REGION_TO_SOURCECOL = {
    "Artery":                   "arteries_n",
    "Vein":                     "veins_n",
    "Vessel (Artery + Vein)":   "both_n",
    "Optic disc":               "optic_disc_n",
    "Optic disc (Temporal)":    "optic_disc_temporal_n",
    "Optic disc (Nasal)":       "optic_disc_nasal_n",
    "Optic disc (Superior)":    "optic_disc_superior_n",
    "Optic disc (Inferior)":    "optic_disc_inferior_n",
}


# -------------------------------------------------------------------------
# Utilities
# -------------------------------------------------------------------------
def _safe_ratio(num, den):
    num, den = pd.to_numeric(num, errors="coerce"), pd.to_numeric(den, errors="coerce")
    ratio = num / den
    ratio[(~np.isfinite(den)) | (den <= 0)] = np.nan
    ratio[~np.isfinite(ratio)] = np.nan
    return ratio


def bootstrap_ci_mean(x, n_boot=2000, ci=95, seed=42):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return (np.nan, np.nan, np.nan, 0)
    rng   = np.random.default_rng(seed)
    boots = np.array([np.nanmean(rng.choice(x, size=x.size, replace=True))
                      for _ in range(n_boot)])
    alpha = (100 - ci) / 2.0
    return (float(np.nanmean(x)),
            float(np.percentile(boots, alpha)),
            float(np.percentile(boots, 100 - alpha)),
            int(x.size))


def format_ci(mean, lo, hi, ref=False):
    if ref:
        return "1.00 (reference)"
    if not (np.isfinite(mean) and np.isfinite(lo) and np.isfinite(hi)):
        return ""
    return f"{mean:.2f} ({lo:.2f} - {hi:.2f})"


def pretty_ft(x):
    return {"lora": "LoRA", "partial": "partial", "linear": "linear"}.get(x, x)


def parse_base_and_ft(desc):
    """Extract base model name and fine-tuning method from a rep folder name.
    Example: 'retfound_dinov2_lora_rank4_ftfull_gradcam' → ('retfound_dinov2', 'lora')
    """
    name   = desc.replace("_gradcam", "")
    tokens = name.split("_")
    for i, t in enumerate(tokens):
        if t in FT_ORDER:
            return "_".join(tokens[:i]) if i > 0 else tokens[0], t
    # fallback: regex search
    m = re.search(r"_(lora|partial|linear)(_|$)", name)
    if m:
        return name[:m.start()].strip("_"), m.group(1)
    return name, "unknown"


def find_quant_csv(desc_folder):
    """Locate RESULTS_test_all_gradcam.csv for a given rep folder name."""
    direct = TENSOR_ROOT / desc_folder / "RESULTS_test_all_gradcam.csv"
    if direct.exists():
        return str(direct)
    all_csvs = glob.glob(str(TENSOR_ROOT / "**" / "RESULTS_test_all_gradcam.csv"), recursive=True)
    if not all_csvs:
        return None
    key = desc_folder.replace("_gradcam", "")
    cands = [p for p in all_csvs if desc_folder in p] or [p for p in all_csvs if key in p]
    return sorted(cands, key=len)[0] if cands else None


# -------------------------------------------------------------------------
# Main
# -------------------------------------------------------------------------
rep_dirs = sorted(p for p in REP_ROOT.glob("*_gradcam") if p.is_dir())
if not rep_dirs:
    raise FileNotFoundError(f"No '*_gradcam' folders found under REP_ROOT={REP_ROOT}")

rows, missing = [], []

for rep_dir in rep_dirs:
    desc        = rep_dir.name
    base_model, ft = parse_base_and_ft(desc)

    quant_csv = find_quant_csv(desc)
    if quant_csv is None:
        missing.append(desc)
        continue

    df = pd.read_csv(quant_csv)
    if REF_COL not in df.columns:
        missing.append(f"{desc} (REF_COL missing)")
        continue

    ref = pd.to_numeric(df[REF_COL], errors="coerce")

    # non-vessel is the reference (ratio = 1)
    m, lo, hi, n = bootstrap_ci_mean(_safe_ratio(ref, ref).values)
    summary = {"Non-vessel": format_ci(m, lo, hi, ref=True)}

    for out_col, src_col in REGION_TO_SOURCECOL.items():
        if src_col not in df.columns:
            summary[out_col] = ""
            continue
        m, lo, hi, n = bootstrap_ci_mean(_safe_ratio(df[src_col], ref).values)
        summary[out_col] = format_ci(m, lo, hi)

    rows.append({"Model": base_model, "Finetuning Method": ft,
                 **summary, "_desc": desc, "_csv": quant_csv})

if not rows:
    raise RuntimeError("No matched quantification CSVs. Check TENSOR_ROOT / REP_ROOT paths.")

out = pd.DataFrame(rows)

out["_ft_rank"]   = out["Finetuning Method"].map(lambda x: FT_ORDER.index(x) if x in FT_ORDER else 999)
out["_base_rank"] = out["Model"].map(lambda x: BASE_MODEL_ORDER.index(x) if x in BASE_MODEL_ORDER else 999)
out = out.sort_values(["_ft_rank", "_base_rank", "_desc"]).reset_index(drop=True)
out["Finetuning Method"] = out["Finetuning Method"].map(pretty_ft)

final_cols = ["Model", "Finetuning Method", "Non-vessel",
              "Artery", "Vein", "Vessel (Artery + Vein)",
              "Optic disc", "Optic disc (Temporal)", "Optic disc (Nasal)",
              "Optic disc (Superior)", "Optic disc (Inferior)"]
final = out[[c for c in final_cols if c in out.columns]].copy()

TENSOR_ROOT.mkdir(parents=True, exist_ok=True)
final.to_csv(SAVE_CSV, index=False)

print(f"[OK] REP folders found : {len(rep_dirs)}")
print(f"[OK] Rows summarized   : {len(final)}")
print(f"[OK] Saved → {SAVE_CSV}")
if missing:
    print("\n[WARN] Could not match the following REP folders:")
    for m in missing:
        print("  -", m)

display(final)